# k-means in production

**← Continues from** [the library lab](kmeans_lab.ipynb): Labs 1–4b fit and judge a k-means model; this notebook runs it
in production. Theory: [k-means README](README.md).

Markdown only, no code: what to store when a k-means model goes live, how to tell that the data has changed, whether
an alarm means re-clustering, and the retraining playbook.

| Part | What it covers |
|---|---|
| [1. Monitor](#monitor) | what to store at go-live, the held-out set, each signal's normal band, signals 1–4 |
| [2. Decide](#decide) | the trigger table: which alarm leads to re-clustering, same k vs different k |
| [3. Retrain](#retrain) | the playbook, from the training window to switching models |
| [4. Profiling](#profiling) | how to describe a cluster, with the wine example |


<a id="monitor"></a>

## 1. Monitor: how do we know something changed?

* Store at go-live, with the model:
  * <span style="color:red">**Before fitting k-means, hold out healthy data:**</span> set aside a random share of the
    training period's rows (e.g. 20%) and never pass them to `KMeans.fit`. 
      - The bands below are built from them. 
      - Bands built from the fitted rows come out too tight, because the centroids were fitted to those very points (fresh
    healthy data gave a far-point share of 1.2%, not training's 1.0%), so the alarms would fire too early.
      - Take random rows of the recent, representative window, not a sparse early phase.
      - Keep at least ~10× the weekly batch size, so the draws below vary honestly.
      - Keep it fixed and compare every batch against it, as NannyML and Evidently do with their reference data.
      - Re-draw it on purpose, never automatically (our procedure; the sources only require a healthy, representative
        reference), from after the change once the behaviour has settled, when:
        - the model is retrained;
        - a confirmed change is accepted without a refit (e.g. a mix shift with the old model kept), or signal 3 fires
          forever;
        - it falls below ~10× the batch size.
      - Optional: add a second, sliding reference (e.g. the last month), which Evidently allows. It catches sudden jumps
        without seasonal false alarms, but on its own it would absorb a slow drift (our reasoning), so keep the fixed one
        too.
  * the monitoring baselines: each feature's training values (signal 1), the training cut-off distance (signal 4), the
    training segment shares (signal 3), and each segment's business metrics;
  * k and its evidence (elbow, silhouette, stability numbers, date, data snapshot), which step 4 compares against later;
  * <a id="normal-band"></a>the normal band of each signal (how far it wobbles by chance), built from the held-out rows:
    1. Draw about 2,000 batches of the weekly batch size (e.g. 1,000 points) from the held-out rows. 
      - If the held-out set is under ~10× the batch size, draw with replacement (bootstrap) so the spread stays right, or use the CLT
       formula below. 
       - Use one row per customer, in the held-out rows and in the weekly batches alike: repeat customers
       make the points correlated, so the real wobble shrinks slower than the band assumes and it comes out too tight.
    2. Score each batch with the stored model and compute signals 1–4 (signal 1: one value per feature, against the
       stored training values).
    3. Per signal (per feature for signal 1), take the 99.5th percentile across the batches: the upper edge of its band.
       Only an upper edge, because for all four signals only higher is bad.
    4. Store the edges. Rebuild them when the model is replaced (step 7), the held-out set is re-drawn, or the weekly
       batch size changes a lot.
    * Example (4 blobs; 500 healthy batches of 1,000 points from 100,000 points the model never saw):

      | signal | typical value (median) | upper edge of the band (99.5th percentile) |
      |---|---|---|
      | 1: KS D (feature 1) | 0.028 | 0.052 |
      | 2: per-point distance | 2.889 | 3.131 |
      | 3: segment-share PSI | 0.002 | 0.012 |
      | 4: far-point share | 1.2% | 2.1% |

    * The bands are much tighter than the rules of thumb (signal 3's edge 0.012 against "0.25 = real shift"), so with
      big batches they also flag real but tiny shifts; to skip those, also require a practical minimum (e.g. PSI > 0.1).
    * <details>
      <summary><b>Why these settings: the math behind the batch size and the number of draws</b></summary>

      * Batch size decides its width: bigger batch, narrower honest band, but later detection.
        * Later detection: a bigger batch needs a longer window (monthly instead of weekly); a change starting early in
          the window shows only when it closes, up to a full window late.
      * Square-root law (Bienaymé, de Moivre): an average of n independent values has spread σ / √n.
      * Signals 2–4 are averages (distances, 0/1 shares): 4× batch halves the wobble; PSI shrinks like 1 / n.

        | batch size | signal 2 wobble (edge − median) | signal 4 wobble (edge − median) | signal 1: KS D (median) | signal 3: PSI (median) |
        |---|---|---|---|---|
        | 250 | 0.484 | 2.2 pp | 0.050 | 0.0096 |
        | 1,000 | 0.224 (halved) | 1.0 pp (halved) | 0.026 (halved) | 0.0026 (quartered) |
        | 4,000 | 0.107 (halved) | 0.46 pp (halved) | 0.013 (halved) | 0.0006 (quartered) |

      * CLT gives the average a bell shape, hence the edge formula: mean + 2.58 × σ / √n; it needs only a mean and a
        spread, so it also works from a small held-out set.
        * 2.58: the 99.5% point of the bell curve.
      * Number of draws decides the edge's precision, not its width; aim for ≥ 10 draws beyond the edge (≈ 2,000 for
        99.5%, hence step 1).
        * 100 draws: 0.5 beyond the edge, wobbling ±141% of itself; edge ±0.030.
        * 500 draws: 2.5 beyond, ±63%; edge ±0.019.
        * 2,000 draws: 10 beyond, ±31%; edge ±0.011.

      </details>
* Signal 1: drift in the input features (KS or PSI per feature).
  * *KS, two-sample Kolmogorov–Smirnov:* compares one feature's training values with today's.
    * D = the largest vertical gap between the two cumulative curves (for every value x, the share of points ≤ x): 0
      means identical, 1 means no overlap. It works on one variable at a time, hence "per feature".
    * It catches any change in the distribution, not just the mean (100,000 values on each side):

      | today vs training | D | p-value |
      |---|---|---|
      | same distribution | 0.004 | 0.41 |
      | mean shifted by 0.05 sd (negligible) | 0.022 | 5 × 10⁻²² |
      | mean shifted by 0.5 sd | 0.196 | ≈ 0 |
      | spread doubled, same mean | 0.165 | ≈ 0 |

    * Read D as the size of the shift and ignore the p-value: at 100,000 points per side any D above about 0.006 is
      "significant", which is why the negligible shift got p = 5 × 10⁻²². D has a plain meaning: D = 0.196 means
      that at some value of the feature, about 20 percentage points more of today's data sits below it.
    * D's threshold is its [band](#normal-band) (one upper edge per feature), not a fixed number like D > 0.1: with no
      change, D shrinks like 1/√n, so a fixed cut-off is too loose for big batches and too strict for small ones.
  * *PSI per feature:* cut the feature into bins (e.g. the training deciles) and compute PSI on the share per bin;
    PSI itself is defined under signal 3.
  * Compute both per segment too, not only on all data: a change inside one segment gets diluted in the overall number
    (0.048 overall vs 0.194 inside the segment, in the example below).
  * <details>
    <summary><b>What to do when signal 1 alarms</b></summary>

    1. Bug or real? Run data checks (schema, ranges, nulls, units) on the column it names. Broken input: fix the data,
       don't retrain.
    2. Real, and signal 2 or 4 is up: their rows in the [trigger table](#decide) decide; signal 1 says which features to
       look at.
    3. Real, and only per-segment signal 1 is up: that segment's shape may have changed (e.g. it split), which signals
       2, 3 and 4 can't see. Run the [k-scan](#k-scan).
    4. Per segment means segments × features tests, so some fire by chance every batch: use a stricter edge and require
       several batches in a row.
    * Example: 4 round blobs (std 1), k = 4 fitted on healthy data; in a batch of 10,000 points per blob, one blob is
      replaced by two tighter sub-blobs side by side along feature x, placed so the mean squared distance to the old
      centroid stays 2.

      | sub-blob std | signal 2 | signal 3 | signal 4 | signal 1: KS on x inside the segment | silhouette k = 4 / k = 5 |
      |---|---|---|---|---|---|
      | no split | 2.003 | 0.0000 | 1.0% | 0.024 | 0.813 / 0.684 |
      | 0.5 | 2.006 | 0.0000 | 0.5% | 0.203 | 0.811 / 0.763 |
      | 0.2 | 2.023 | 0.0000 | 0.6% | 0.330 | 0.815 / **0.819** |

      Signals 2–4 stay flat; only signal 1 fires, and at std 0.2 the k-scan prefers k = 5. Built to keep signal 2 flat,
      so it shows the case can exist, not how often; a split along a diagonal (weaker per feature) is untested.

    </details>
* Signal 2: WCSS per point of the training sample vs WCSS per point of the new-data. (inertia (i.e. WCSS) ÷ number of points, so
  it doesn't grow with volume, because naturally WCSS will grow since a non-negative quantity is being added to a non-negative quantity).
  * **Alarm** when it's above its [band](#normal-band), not when it rises against training's value: fresh healthy data
    already sits higher (2.92 vs 2.86), because the centroids were fitted to the training points.
  * **Why it jumps:** an average; 80% old points ≈ 2.9, 20% new-group points ≈ 33.6 (no centroid of their own) →
    ≈ 9.0, +216%.
  * **Blind spot:** small or nearby new group → small jump (nearby 20%: only +92%); signal 4 counts far points
    instead → covers it.
  * Example (also used under signal 3): 4 round blobs, 10,000 points per batch, scored with the stored centroids.

    | batch | signal 2 |
    |---|---|
    | training period | 2.86 |
    | nothing changed | 2.92 (+2%) |
    | 20% of points form a new group | **9.08 (+218%)** |

* Signal 3: segment shares (PSI).
  * *PSI, population stability index:* one number for how much a set of shares changed.
    * PSI = Σ over categories of (today's share − training share) × ln(today's share ÷ training share); every term is
      positive whether a share grew or shrank, so changes don't cancel. \
      **Note:** Today's share meaning incoming samples who are predicted using the currently active k-means centroids.
    * Segments are categories, so their shares fit it directly.
    * Worked example, training 25 / 25 / 25 / 25 % against today's 10 / 10 / 10 / 70 %: each shrunk segment gives
      (0.10 − 0.25) × ln(0.10 / 0.25) = 0.137, three of them 0.412; the grown one gives (0.70 − 0.25) × ln(0.70 / 0.25)
      = 0.463; PSI = 0.412 + 0.463 ≈ 0.88.
    * Rule of thumb from credit scoring: below 0.1 stable, 0.1–0.25 worth watching, above 0.25 a real shift.
      Don't alarm on it: with no change, the PSI's spread is about χ²(k − 1)/n, so its cut-off depends on the batch
      size and the number of segments (du Pisanie et al. 2023); fixed cut-offs fire too often on small batches and
      miss shifts on big ones. Use the [band](#normal-band).
    * A share of 0 makes the log undefined: replace zeros with a tiny value (e.g. 0.0001).
  * Example (same setup as signal 2):

    | batch | segment shares | signal 3 (PSI) |
    |---|---|---|
    | training period | 25 / 25 / 25 / 25 % | — |
    | nothing changed | 25 / 25 / 25 / 25 % | 0.00 |
    | (incoming samples) customers shift between segments | 10 / 10 / 10 / 70 % | **0.88** |

  * <details>
    <summary><b>What to do when signal 3 alarms (it never decides re-clustering on its own)</b></summary>

    * Confirm it's a pure mix shift: signals 2 and 4 stay inside their [bands](#normal-band). If either is up, their rows
      in the [trigger table](#decide) decide, not signal 3.
    * Keep the model: customers moved between segments but sit near the old centroids, so the segments are still right
      and only their sizes changed; a same-k refit would move the centroids no more than refit noise.
    * Report it to the business: segment sizes drive campaign budgets and capacity.
    * Once the new mix is accepted, re-draw the held-out set and reset the training shares, or signal 3 stays above its
      band every week.

    </details>

* Signal 4: the share of new points whose distance to their predicted centroid exceeds the training cut-off (the 99th
  percentile of the training points' distances to their own centroids; 1% of training points exceed it by definition).
  * **Alarm** when it's above its [band](#normal-band) (2.1% at 1,000-point batches), not above 1%: fresh healthy data
    already gives 1.2%.
  * Example (same setup as signal 2):

    | batch | signal 4 |
    |---|---|
    | training period | 1.0% |
    | nothing changed | 1.2% |
    | 20% of points form a new group | **20.7%** |

* The monitoring job and who brings what:
  * *A monitoring job, on every new batch of data (daily or weekly):* assign the batch with the stored model's
    `predict`, compute signals 1–4, compare each with its baseline, and log them to a dashboard. \
    Running the *checks* on a schedule is fine; only *retraining* waits for evidence.
  * *Who brings what:*
    * the monitoring job: signals 1–4;
    * the teams using the segments (marketing, CRM): concept drift, e.g. "coupon redemption in deal hunters fell from
      18% to 6%"; no data signal shows it;
    * the data scientist owns the thresholds, reads the alarms, and makes the call with the trigger table.
* <span style="color:red">**TODO (not done):**</span> The concept-drift case, which only people can report: business
  metrics per segment stop responding.
  * **Setup:** an online shop clusters customers on purchase frequency, basket size, discount usage and days since the
    last order. One segment, the "deal hunters", gets coupon campaigns, and 18% redeem them.
  * **The change:** a competitor launches free-shipping membership. The deal hunters still shop just as often and
    still use discounts, but they now chase free shipping, so coupon redemption drops to 6%.
  * **What the monitors show:**
    * the features haven't moved, so KS and PSI per feature are flat;
    * new customers sit just as close to the centroids, so the distance signal is flat;
    * the segment shares haven't moved, so their PSI is flat.
  * **What changed is the meaning.** "This segment responds to coupons" stopped being true, while the data looks the
    same. Only the business metric per segment (coupon redemption) reveals it.
  * **The fix usually isn't a plain re-fit:** you need a new feature, such as sensitivity to shipping costs, and then
    you re-cluster (step 4's first scenario continues this story).


<a id="decide"></a>

## 2. Decide: does this alarm mean re-cluster?

One row per trigger, with its threshold. **Same k:** step 5 always makes the decision; this table says what to watch
for. **Different k:** step 4 decides with the real × meaningful table, starting with the check named here. Bold rows:
the trigger leads to re-clustering.

| trigger that fired (threshold) | same k: what a refit typically shows | same k: what to watch for | different k: how k usually changes | different k: what [step 4](#retrain-step-4) should check first |
|---|---|---|---|---|
| **signal 2**: mean squared distance above its [normal band](#normal-band) | **centroids moved** | **the new centroids should bring the distance back into its [band](#normal-band); if it stays high, same-k k-means can't absorb the change (step 3)** | **either way** | **whether the gain is real (95% rule, time windows) before anything else; often a same-k refit fixes it** |
| **signal 4**: far-point share above its [normal band](#normal-band) (a new group) | **the scan may still prefer the same k, because the silhouette can miss a faint new group** | **don't simply adopt: the new group is being forced into an old segment; [profile](#profiling) that segment and test k + 1 against the business** | **k + 1** | **the contingency table: the new cluster should be mostly new points, not a split of an old segment** |
| **signal 1, per segment**: a feature's distribution changed inside one segment (not a data bug) | **the centroid barely moves** | **the scan may keep k; [profile](#profiling) the segment and test k + 1 against the business** | **k + 1** | **the contingency table: the new clusters should be a clean split of that segment** |
| **a business metric per segment falls** (concept drift; reported by the teams) | **same features, so nearly the same centroids** | **a refit won't help; it needs new features, which means a new model, not new centroids** | **often changes, after new features are added** | **meaningful first: does the new segmentation fix the business metric in the holdout?** |
| **an upstream feature is added, removed, redefined or rescaled** | **the old centroids no longer fit the new feature space** | **retraining is forced; carry names over by matching [profile cards](#profiling), not centroid distances, since the spaces differ** | **may change** | **match old and new segments by [profile cards](#profiling), since the feature spaces differ** |
| **the business changes the k budget: how many segments it can act on (e.g. a new team or tool lets marketing run 8 distinct campaigns instead of 5; a budget cut leaves room for 3)** | **—** | **not a same-k case: k changes by definition** | **fixed by the business** | **skip the k-scan (k is given); check per-cluster stability, "meaningful?" and the name mapping ([when the business fixes k](#k-budget))** |


<a id="retrain"></a>

## 3. Retrain: a playbook, run in this order once the trigger table's decision is to re-cluster

* **Step 1 · Choose the training window by backtesting, not by gut.**
  * No drift detected: use all the data; more data gives a more stable fit.
  * Drift detected:
    * Find when it started: compute the monitoring signals (distance, segment-share PSI) per week over the history;
      the change point is the first week they left their [normal band](#normal-band).
    * Line up candidate windows: all data, since the change point, the last 6 months, the last 3 months.
    * Fit k-means on each window and measure its mean squared distance on the most recent **held-out** week.
    * Keep the window whose model fits that week best, provided it has enough rows per cluster (FAISS's default
      floor is 39 points per centroid).
  * Compute each customer's features over a rolling period (the last N days), so a customer whose behaviour changed
    is described by the new behaviour, not the old.
* <a id="k-scan"></a>**Step 2 · Refit k-means with the full k-scan** on a sample of that window: elbow, silhouette, stability and gap
  together. Refit the scaler and PCA with it; they are part of the model.
* **Step 3 · Check that k-means still fits.** Three checks on recent data; if any fails, review the method:
  * *Fit recovered:* the new centroids' mean squared distance is back inside its [normal band](#normal-band) (the range
    seen on past healthy weeks).
  * *Far points gone:* the share of points beyond training's 99th-percentile distance is back near 1%.
  * *No hidden shapes:* HDBSCAN on a sample finds no groups that are stable across subsamples yet spread over several
    k-means clusters in the contingency table.
  * *Method review, only if a check failed.* Compare the candidate methods on:
    * *Agreement* (ARI between two methods' labels): a consistency check, not a ranking. 
      * High means two methods with different assumptions see the same structure; low means at least one is wrong, and it can't say which.
    * *Stability of each method:* fix each method's settings (k, `min_cluster_size`) across ~30 subsamples and
      compare the runs' labels by ARI. 
      - DBSCAN and HDBSCAN have no `predict`, so compare two runs on the points both
      subsamples contain, and report the noise share beside the ARI. 
      - Stable is necessary, not sufficient: k-means cuts the moons the same wrong way every time.
    * *Shape:* several 2-D views (not just PC1–2) and the explained-variance spectrum.
    * *Not internal scores alone:* silhouette, Calinski–Harabasz and Davies–Bouldin reward round clusters, so they
      favour k-means by construction.
    * *The business holdout decides* (step 6).
* <a id="retrain-step-4"></a>**Step 4 · If the scan picks a different k: is the new k real and meaningful?**
  Start with the check the trigger table (section 2) names for the trigger that fired.
  * *Real?* On about 30 subsamples (80% of the rows each, no row twice within one), fit both k on the same subsample;
    the new k must have the higher silhouette on at least 95% of them. 
    - Report the gain's median and 5th–95th percentile, e.g. +0.022 (+0.018 to +0.026). 
    - It should also win in most recent time windows (each of the last 4–6 weeks scanned separately); without timestamps skip this, and another split (country, channel) can play the same
    role. 
    - Finally, the old-vs-new contingency table should show a clean split or merge, not a reorganisation (below).
  * *Meaningful?* The new clusters' profiles differ ([profiling recipe](#profiling) below), their business outcomes
    differ, and the holdout in step 6 beats the old segmentation.

  | real? | meaningful? | decision |
  |---|---|---|
  | yes | yes | adopt the new k; map names, splitting the old name where a segment split |
  | yes | no | keep the old k for actions; track the new structure as sub-segments |
  | no | — | false alarm: keep the old k |

  * *Why follow the table:*
    * Better clustering metrics alone can't justify a change: with big data even a tiny gain wins on every subsample,
      and the silhouette can miss a real change entirely.
    * The old k isn't sacred either: the data really can change.
    * Switching costs something (dashboards, campaign rules, people's vocabulary), so it needs evidence on both
      hurdles.
    * It turns "why did you change the segments?" into an answer stakeholders can check.
  * *Scenarios (online shop):*
    * **Yes, yes:** the sequel to the concept-drift case in section 1. After a shipping-sensitivity feature is added,
      the re-cluster splits the "deal hunters" into coupon responders and shipping-sensitive buyers. k = 6 beats
      k = 5 on 100% of subsamples, holds for 6 weeks, and is a clean split of one segment; in a holdout,
      free-shipping offers to the new segment lift conversion from 4% to 7%. Adopt k = 6, naming the halves "deal
      hunters: coupons" and "deal hunters: shipping".
    * **Yes, no:** a new "sessions on mobile" feature splits the "regulars" into app and web users, stable and
      persistent, but both halves have the same basket, campaign response and churn. Keep k = 5 for actions; track
      app vs web as sub-segments.
    * **No:** in Black Friday week a flood of one-off bargain shoppers forms a temporary cluster, and a 6th cluster
      wins that week's scan; two weeks later it is gone. False alarm: keep k = 5.
  * *Reading the old-vs-new contingency table* (rows: old segments, columns: new clusters, cells: customers in both):
    * Example: old k = 3 vs new k = 4, the same 300 customers labelled by both models.

      | | new 0 | new 1 | new 2 | new 3 |
      |---|---|---|---|---|
      | **old A** | 100 | 0 | 0 | 0 |
      | **old B** | 0 | 52 | 48 | 0 |
      | **old C** | 0 | 0 | 0 | 100 |

      * Old B split into new 1 and new 2, and every new cluster draws its members from one old segment: a clean split.
      * Had old B's customers landed across new 0, 1 and 2 alongside other segments' customers, it would be a
        reorganisation.
    * Clean split: each new cluster draws almost all its members from one old segment.
    * Clean merge: each old segment goes almost entirely into one new cluster.
    * Reorganisation: members spread across several old segments.
    * Don't judge by ARI when k differs: a k change forces disagreement, so a tidy refinement can score as low as a
      messy reshuffle (10,000 customers in 4 segments of 2,500):

    | old k → new k | what happened | ARI |
    |---|---|---|
    | 4 → 5 | one segment split cleanly in half | 0.913 |
    | 4 → 8 | every segment split cleanly in half | 0.600 |
    | 4 → 3 | two segments merged cleanly | 0.714 |
    | 4 → 4 | 3.8% of customers switched segment | 0.902 |
    | 4 → 4 | 15% of customers switched segment | 0.641 |

  * <a id="k-budget"></a>*When the business fixes k (5 → 8 or 8 → 5):*

    | | 5 → 8 (more segments) | 8 → 5 (fewer segments) |
    |---|---|---|
    | statistical risk | unstable splits of noise | low |
    | business risk | paying for campaigns with no extra lift | erasing a profitable distinction |
    | what the DS brings | which splits are real, and what each extra segment is worth | a merge plan with each merge's cost |
    | evidence | per-cluster stability on subsamples; [profiles](#profiling); holdout lift of the new campaigns vs the old ones | outcome similarity per pair of segments (response, value, churn); fit lost per merge; contingency table showing a clean merge, not a reorganisation; holdout of a merged campaign vs the separate ones |
    | ways to use the capacity | sub-split only the segments whose halves really differ (two levels); spend extra campaigns on in-segment A/B tests; add features that might reveal real distinctions | merge by outcome similarity, not centroid closeness; keep a valuable distinction inside a merged segment through cheaper means (different message or offer rules, not a separate campaign); move the least valuable segment to a default "no campaign" group instead of merging it |
    | if the business still insists | deliver 8, flag the segments that aren't distinct, and let outcomes per segment decide at the next review | do their merge, state its price, and keep the old 8 labels as tags inside the merged segments so the loss is measured and the merge can be undone |

* **Step 5 · If the scan picks the same k: keep, replace silently, or replace visibly.**
  * Definitions, all on recent data after matching each new centroid to its nearest old one:
    * *shift:* distance between a matched old and new centroid;
    * *noise:* how far that centroid moves across refits on subsamples of the recent data;
    * *fit gain:* (old centroids' mean squared distance ÷ new centroids' mean squared distance) − 1, on held-out
      recent data;
    * *fit noise:* how much the fit gain varies across those refits;
    * *switch share:* the share of recent customers whose segment changes.
  * Decide in this order:
    * Refits disagree among themselves (low ARI between them): don't replace; investigate the instability.
    * No centroid's shift exceeds its noise: keep the old model.
    * Fit gain within the fit noise: keep the old model.
    * Switch share ≈ 0: replace silently, under the same names.
    * Switch share > 0: replace visibly; [re-profile](#profiling) the moved segments, tell the teams, watch the business
      metrics.
  * Cluster spread is not a decision input; use it only to explain size to stakeholders ("the centroid moved 1.3×
    its own radius"). With the same k, report the switch share rather than the ARI: same information, easier to
    explain.
* **Step 6 · Validate before switching.**
  * The fit on held-out recent data, never on the data the model was trained on.
  * A holdout experiment on live customers: randomly split the customers active during the test period into
    control (actions from the old segments) and treatment (actions from the new ones), and measure the business
    metric afterwards. Historical points can't be used: the outcomes have to follow the actions.
* **Step 7 · Switch carefully.**
  * Carry names over. Labels are arbitrary per fit (two fits of the same data swapped labels 0 and 2), so without
    this, campaign rules, dashboards and CRM tags would silently point at different customers. Match each new centroid
    to its nearest old one, or match [profile cards](#profiling) if the feature space changed; only new or split
    segments get new names.
  * [Re-profile](#profiling) the segments.
  * Version the old model and keep a rollback.
  * Tell the teams whose rules use the segments.
  * Reset the monitoring baselines ([distance band](#normal-band), 99th-percentile distance, segment shares for PSI), or
    the alarms keep firing or stay silent.


<a id="profiling"></a>

## 4. Reference: profiling a cluster

Used in steps 4, 5 and 7. For each cluster:

1. *Size:* count and share of all rows.
2. *Centre in original units:* the median of each feature over the cluster's members (€, orders a year), not the
   scaled values.
3. *Distinguishing features:* (cluster mean − overall mean) / overall standard deviation for each feature; keep the
   3 largest in absolute value; they drive the name. For a categorical feature use lift: the category's share in
   the cluster ÷ its share overall.
4. *How solid it is:* the average silhouette of its members; a low value flags a blurry cluster.
5. *Business outcomes:* conversion, revenue, churn per cluster, with confidence intervals, on variables the
   clustering never saw.
6. *Examples:* the 3–5 members closest to the centroid, plus a couple of borderline ones.
7. *Readable rules (optional):* a shallow decision tree predicting the cluster from the features; its accuracy
   shows how faithful the rules are.
* *If no sensible name comes:* treat it as a warning, not proof; use a descriptive name ("frequent, small basket,
  app") or an ID with its profile card. No distinct profile and no business difference: real but not meaningful.
* **Worked example to code later: wine** (`load_wine`, 13 features, standardised, `KMeans(3, n_init=10,
  random_state=0)`; recipe steps 1–4, 6, 7; no business outcomes in this dataset):

  | cluster | size | silhouette | distinguishing features (median, difference from average) |
  |---|---|---|---|
  | 0 | 65 (37%) | 0.18 | alcohol 12.25 (−0.9 sd), colour intensity 2.90 (−0.9 sd), proline 495 (−0.8 sd) |
  | 1 | 51 (29%) | 0.35 | OD280/OD315 1.68 (−1.3 sd), flavanoids 0.70 (−1.2 sd), hue 0.67 (−1.2 sd) |
  | 2 | 62 (35%) | 0.34 | proline 1,072 (+1.1 sd), flavanoids 2.98 (+1.0 sd), total phenols 2.83 (+0.9 sd) |

  * Names from the features: 2 = rich, high-proline, high-phenol; 0 = light, low-alcohol, pale; 1 = low-flavanoid.
  * Cluster 0's silhouette (0.18) is a warning: loosely grouped, so its name needs care.
  * Most typical rows (closest to the centroid): cluster 0: 106, 117, 81; cluster 1: 148, 174, 163; cluster 2: 48,
    57, 35.
  * Depth-2 decision tree (`DecisionTreeClassifier(max_depth=2, random_state=0)` on the raw features), reproducing
    the clusters for 94% of the wines:
    * proline > 755 and flavanoids > 2.17 → cluster 2
    * proline > 755 and flavanoids ≤ 2.17 → cluster 1
    * proline ≤ 755 and OD280/OD315 ≤ 2.19 → cluster 1
    * proline ≤ 755 and OD280/OD315 > 2.19 → cluster 0
